In [7]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split # use for data distribution for clients

In [4]:
# Data Loading
transform = transforms.ToTensor()
mnist_trainset=datasets.MNIST(root = './data', train = True, download = True, transform = transform)
mnist_testset=datasets.MNIST(root = './data', train = False, download = True, transform = transform)
image, label = mnist_trainset[0]

In [6]:
# CNN
# a CNN with 2 layers: Layer1-> Relu->Pool->Layer2->...->Flatten->Fully connected->Output
class CNN(nn.Module): # we take the standard class and modify it
    def __init__(self): # this is a constructor
        super(CNN,self).__init__()
        # First layer
        self.conv1 = nn.Conv2d(in_channels=1,   # Mnist has 1 channel
                          out_channels=16, # 16 filters
                          kernel_size=3)   # 3x3 kernel size
        # Second layer
        self.conv2 = nn.Conv2d(in_channels=16,   # Mnist has 1 channel
                          out_channels=32, # 16 filters
                          kernel_size=3)   # 3x3 kernel size
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        # create the fully connected layer
        self.fully_connected = nn.Linear(800,10)

    def forward(self,x):
        x = self.pool(self.relu(self.conv1(x))) # do relu and pooling for layer 1
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # here we reshape the tensor to be of 'out.size(0)' rows 
                                  # but don't know exactly how many columns, so we specify '-1'
                                  # [1, 32, 5, 5] -> [1, 800]
        x = self.fully_connected(x)
        return x

model = CNN()
out = model(image.unsqueeze(0))
print(out.shape)

torch.Size([1, 10])


In [12]:
# Client abstraction, split dataset into clients
number_of_clients = 5

dataset_size = len(mnist_trainset)
client_size = dataset_size // number_of_clients # we use // instead of / to remove the fractional part
client_datasets = random_split(mnist_trainset, [client_size]*number_of_clients) # client_datasets[0], client_datasets[1], ..., etc.

client_loaders = []
for dataset in client_datasets:
    loader = DataLoader(dataset, batch_size = 64, shuffle = True)
    client_loaders.append(loader)

print(len(client_loaders))
print(len(client_loaders[0]))
images, labels = next(iter(client_loaders[0]))
print(images.shape)

5
188
torch.Size([64, 1, 28, 28])
